# Quantize FLUX.2 [klein] 4B to GGUF

Turn the 7.75 GB `flux-2-klein-4b.safetensors` into small GGUF files (Q8_0 about 4.3 GB, Q4_0 about 2.5 GB) that run on a plain CPU with [stable-diffusion.cpp](https://github.com/leejet/stable-diffusion.cpp).

Quantizing needs **RAM and disk, not a GPU**. A free Kaggle CPU session is enough.

**Before you run**
1. Right sidebar -> Session options -> **Internet: On**
2. Optional: add a Kaggle Secret named `HF_TOKEN` (a Hugging Face token). Every repo used here is public, so this only avoids anonymous rate limits.
3. Run All.

**You get:** `klein4b_bundle/` and `klein4b_bundle.zip` in the Output tab with the quantized model(s), the Qwen3-4B text encoder, the VAE, and `SHA256SUMS.txt`.

## 1. Config

In [ ]:
import os

MODEL_REPO = "black-forest-labs/FLUX.2-klein-4B"
MODEL_FILE = "flux-2-klein-4b.safetensors"

VAE_REPO = "Comfy-Org/flux2-klein-4B"
VAE_FILE = "split_files/vae/flux2-vae.safetensors"

LLM_REPO = "unsloth/Qwen3-4B-GGUF"
LLM_FILE = "Qwen3-4B-Q4_K_S.gguf"

QUANT_TYPES = ["q8_0", "q4_0"]

RUN_TEST = False
TEST_PROMPT = "a cat sitting on a sofa, photo"

PUSH_TO_HF_REPO = None

WORK = "/kaggle/working"
TMP = "/kaggle/temp"
OUT = f"{WORK}/klein4b_bundle"
SD_DIR = f"{WORK}/stable-diffusion.cpp"

os.makedirs(TMP, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

`QUANT_TYPES` accepts `f16`, `q8_0`, `q5_0`, `q5_1`, `q4_0`, `q4_1`. Higher bits means better quality and a bigger file. Set `PUSH_TO_HF_REPO` to e.g. `"your-name/flux2-klein-4b-gguf"` (needs an `HF_TOKEN` with write access) to upload the result to your own Hugging Face repo instead of downloading from Kaggle.

## 2. Build stable-diffusion.cpp (CPU)

In [ ]:
%%bash
set -e
cd /kaggle/working
if [ ! -d stable-diffusion.cpp ]; then
  git clone --recursive https://github.com/leejet/stable-diffusion.cpp
fi
cd stable-diffusion.cpp
cmake -B build -DCMAKE_BUILD_TYPE=Release
cmake --build build --config Release -j"$(nproc)"
ls build/bin

In [ ]:
SD = next((p for p in (f"{SD_DIR}/build/bin/sd-cli", f"{SD_DIR}/build/bin/sd") if os.path.exists(p)), None)
assert SD, "Build failed: no sd-cli binary found. Check the build output above."
print("Using:", SD)

## 3. Hugging Face login (optional)

In [ ]:
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret("HF_TOKEN"))
    print("Logged in with HF_TOKEN")
except Exception as e:
    print("Continuing anonymously:", e)

## 4. Download (about 11 GB total, stored in /kaggle/temp)

In [ ]:
import shutil
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(MODEL_REPO, MODEL_FILE, local_dir=TMP)
vae_path = hf_hub_download(VAE_REPO, VAE_FILE, local_dir=TMP)
llm_path = hf_hub_download(LLM_REPO, LLM_FILE, local_dir=TMP)

vae_name = os.path.basename(VAE_FILE)
shutil.copy(vae_path, f"{OUT}/{vae_name}")
shutil.copy(llm_path, f"{OUT}/{LLM_FILE}")

for p in (model_path, vae_path, llm_path):
    print(f"{os.path.getsize(p) / 1e9:6.2f} GB  {p}")

## 5. Quantize

In [ ]:
import subprocess

gguf_paths = {}
for q in QUANT_TYPES:
    out = f"{OUT}/flux-2-klein-4b-{q}.gguf"
    cmd = [SD, "-M", "convert", "-m", model_path, "-o", out, "--type", q]
    print("Running:", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0 or not os.path.exists(out):
        print(f"FAILED {q}\n{r.stdout[-1500:]}\n{r.stderr[-1500:]}")
        print("If flags were rejected, run: !" + SD + " --help")
        continue
    with open(out, "rb") as f:
        assert f.read(4) == b"GGUF", f"{out} is not a valid GGUF file"
    gguf_paths[q] = out
    print(f"OK {q}: {os.path.getsize(out) / 1e9:.2f} GB")

assert gguf_paths, "No quantized file was produced."

## 6. Test image (optional)

In [ ]:
if RUN_TEST:
    q = next(iter(gguf_paths))
    test_png = f"{OUT}/test-{q}.png"
    cmd = [SD,
           "--diffusion-model", gguf_paths[q],
           "--llm", f"{OUT}/{LLM_FILE}",
           "--vae", f"{OUT}/{vae_name}",
           "-p", TEST_PROMPT,
           "--cfg-scale", "1.0", "--steps", "4", "--diffusion-fa",
           "-W", "512", "-H", "512", "-t", str(os.cpu_count()),
           "-o", test_png]
    print(" ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
    if os.path.exists(test_png):
        from IPython.display import Image, display
        display(Image(test_png))
else:
    print("Skipped. Set RUN_TEST = True in the config cell to enable.")

## 7. Package

In [ ]:
import hashlib

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 24), b""):
            h.update(chunk)
    return h.hexdigest()

names = sorted(n for n in os.listdir(OUT) if n != "SHA256SUMS.txt")
with open(f"{OUT}/SHA256SUMS.txt", "w") as f:
    for n in names:
        f.write(f"{sha256(f'{OUT}/{n}')}  {n}\n")

shutil.make_archive(f"{WORK}/klein4b_bundle", "zip", OUT)

total = 0
for n in names:
    size = os.path.getsize(f"{OUT}/{n}") / 1e9
    total += size
    print(f"{n:45s} {size:6.2f} GB")
print(f"{'TOTAL':45s} {total:6.2f} GB")

## 8. Upload to Hugging Face (optional)

In [ ]:
if PUSH_TO_HF_REPO:
    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(PUSH_TO_HF_REPO, exist_ok=True)
    api.upload_folder(folder_path=OUT, repo_id=PUSH_TO_HF_REPO)
    print("Uploaded to https://huggingface.co/" + PUSH_TO_HF_REPO)
else:
    print("Skipped. Download klein4b_bundle.zip from the Output tab (right sidebar).")

## Run it locally

Get a prebuilt `sd-cli` from the [stable-diffusion.cpp releases](https://github.com/leejet/stable-diffusion.cpp/releases), then:

```
./sd-cli --diffusion-model flux-2-klein-4b-q4_0.gguf \
  --llm Qwen3-4B-Q4_K_S.gguf \
  --vae flux2-vae.safetensors \
  -p "your prompt" --cfg-scale 1.0 --steps 4 --diffusion-fa -W 512 -H 512 -o out.png
```